# INOVAAPPS 2026 · Sinais relevantes

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display

from dados import CORES, ORDEM_STATUS, carregar, fmt_brl
from sinais import (FORMAS, ROTULO_FORMA, SINAIS, Limiar, Regra, antecedencia_alerta, desempenho, disparos,
                    episodios, episodios_sinal, evidencias, historico, tabela_auc, valores_ate)

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates["inova"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Segoe UI, system-ui, sans-serif", size=13, color="#2b2b29"),
    title=dict(font=dict(size=17), x=0.01, xanchor="left"),
    separators=",.", paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
    hoverlabel=dict(bgcolor="#ffffff", bordercolor="#c9c7c0", font=dict(size=12, color="#2b2b29")),
    legend=dict(orientation="h", yanchor="bottom", y=1.06, xanchor="right", x=1, title_text=""),
    margin=dict(l=70, r=30, t=110, b=60),
    xaxis=dict(gridcolor="#eceae5", linecolor="#bdbbb4", zeroline=False),
    yaxis=dict(gridcolor="#eceae5", linecolor="#bdbbb4", zeroline=False),
))
pio.templates.default = "plotly_white+inova"
CINZA, CINZA_CLARO, TINTA = "#8a8984", "#eeede9", "#2b2b29"
AMBAR = "#d68a00"
COR_SINAL = dict(zip(SINAIS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]))
ROT = {s: c.rotulo for s, c in SINAIS.items()}

def rgba(hex_, a):
    h = hex_.lstrip("#")
    return f"rgba({int(h[:2], 16)},{int(h[2:4], 16)},{int(h[4:], 16)},{a})"

def br(v, casas=2):
    return f"{v:.{casas}f}".replace(".", ",").replace("-", "−")

def pct(n, tot):
    return f"{100 * n / tot:.0f}% ({int(n)}/{tot})"

def sub(txt):
    return f"<br><span style='font-size:12px;color:{CINZA}'>{txt}</span>"

d = carregar()
cli = d.clientes.set_index("cliente_id")
KS = range(1, 16)
V = {k: valores_ate(d, k) for k in KS}          # valores por cliente usando só dados até o mês −k
ST = V[1]["status"]
N_CANC, N_ATIV = int((ST == "Cancelou").sum()), int((ST == "Ativo").sum())
assert (N_CANC, N_ATIV) == (22, 58)

## 1. Nível ou variação

In [2]:
auc = tabela_auc(d, range(1, 7))
m13 = auc[auc.k <= 3].groupby(["sinal", "forma"]).auc.mean().unstack()
MELHOR = {s: "variacao" if m13.loc[s, "variacao"] > m13.loc[s, "nivel"] + 0.01 else "nivel" for s in SINAIS}
ORDEM = sorted(SINAIS, key=lambda s: -m13.loc[s, MELHOR[s]])

fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.03,
                    subplot_titles=[f"Dados até o mês −{k}" for k in (1, 2, 3)])
estilo = {"nivel": dict(color=CINZA, symbol="circle"), "variacao": dict(color="#1c5cab", symbol="diamond")}
ylab = [f"{ROT[s]}<br><span style='font-size:11px;color:{CINZA}'>melhor: {ROTULO_FORMA[MELHOR[s]].lower()}</span>" for s in ORDEM]
for c, k in enumerate((1, 2, 3), start=1):
    q = auc[auc.k == k].set_index(["sinal", "forma"])
    for s, y in zip(ORDEM, ylab):
        fig.add_trace(go.Scatter(x=[q.loc[(s, f), "auc"] for f in FORMAS], y=[y, y], mode="lines",
                                 line=dict(color="#d6d4ce", width=3), hoverinfo="skip", showlegend=False), row=1, col=c)
    for f in FORMAS:
        qq = q.xs(f, level="forma").loc[ORDEM]
        fig.add_trace(go.Scatter(x=qq.auc, y=ylab, mode="markers", name=ROTULO_FORMA[f] + (" (média dos 3 últimos meses)" if f == "nivel" else " (3 últimos meses − média anterior do cliente)"),
                                 legendgroup=f, showlegend=c == 1, marker=dict(size=12, line=dict(color="white", width=1.5), **estilo[f]),
                                 customdata=np.c_[[ROT[s] for s in ORDEM], qq.n_canc],
                                 hovertemplate=f"<b>%{{customdata[0]}}</b> · {ROTULO_FORMA[f]} · até −{k}<br>AUC %{{x:.2f}}"
                                               "<br>%{customdata[1]} de 22 cancelados com dado<extra></extra>"), row=1, col=c)
fig.update_xaxes(range=[0.45, 1.02], dtick=0.1, tickformat=".1f", title_text="AUC")
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=520, margin=dict(l=190, t=150),
                  title="Nível ou variação? A queda do uso separa melhor com antecedência; nos demais sinais o nível basta"
                        + sub("AUC de Cancelou × Ativo · “melhor” = maior AUC média entre −1 e −3 (variação só vence se ganhar por > 0,01)"))
fig.show()
display(m13.rename(columns=ROTULO_FORMA, index=ROT).round(3).rename_axis(index="AUC média (−1 a −3)", columns=None))

,Nível,Variação
AUC média (−1 a −3),,
Atraso de pagamento,0.729,0.709
Chamados críticos,0.804,0.792
Nota NPS,0.881,0.887
Reclamações formais,0.787,0.765
SLA cumprido,0.807,0.794
Tempo médio de resolução,0.839,0.803
Uso da plataforma,0.911,0.924


In [3]:
u = V[1]["uso"].join(ST.rename("status"))
XL, YL = 70, -10
fig = go.Figure()
for st in ORDEM_STATUS:
    s = u[u.status == st]
    fig.add_trace(go.Scatter(x=s.nivel, y=s.variacao, mode="markers", name=f"{st} ({len(s)})", marker_color=CORES[st],
                             marker=dict(size=10, opacity=0.8, line=dict(color="white", width=1)),
                             customdata=np.c_[s.index, s.base],
                             hovertemplate=f"<b>%{{customdata[0]}}</b> · {st}<br>Uso nos 3 últimos meses: %{{x:.1f}}%"
                                           "<br>Média anterior: %{customdata[1]:.1f}%<br>Variação: %{y:+.1f} p.p.<extra></extra>"))
fig.add_vline(x=XL, line=dict(color=CINZA, dash="dot", width=1.5))
fig.add_hline(y=YL, line=dict(color=CINZA, dash="dot", width=1.5))
quad = [("nivel <= @XL and variacao <= @YL", 45, -32, "Baixo e caindo"), ("nivel > @XL and variacao <= @YL", 99, -32, "Alto, mas caindo"),
        ("nivel <= @XL and variacao > @YL", 45, 17, "Baixo e estável"), ("nivel > @XL and variacao > @YL", 99, 17, "Alto e estável")]
for q, x, y, nome in quad:
    n = u.query(q).status.value_counts()
    fig.add_annotation(x=x, y=y, showarrow=False, xanchor="left" if x < 60 else "right", align="left" if x < 60 else "right",
                       text=f"<b>{nome}</b><br><span style='color:{CORES['Cancelou']}'>Cancelou {n.get('Cancelou', 0)}</span> · "
                            f"<span style='color:{CORES['Ativo']}'>Ativo {n.get('Ativo', 0)}</span>", font_size=12)
baixos = u[(u.status == "Ativo") & (u.nivel <= 75)]
fig.update_xaxes(title_text="Uso médio nos 3 últimos meses (%)", range=[44, 100], ticksuffix="%")
fig.update_yaxes(title_text="Variação vs média anterior do cliente (p.p.)", range=[-34, 20])
fig.update_layout(height=560, title="Uso sempre baixo ≠ uso caindo: quem cancelou despenca contra o próprio histórico"
                  + sub(f"Dados até o mês −1 · linhas: uso ≤ {XL}% e queda ≥ {-YL} p.p. · {len(baixos)} ativos com uso ≤ 75%, "
                        f"dos quais {(baixos.variacao > -5).sum()} sem queda relevante (> −5 p.p.)"))
fig.show()

## 2. Antecedência sem vazamento

In [4]:
best = pd.concat([auc[(auc.sinal == s) & (auc.forma == MELHOR[s])] for s in SINAIS])
def lead_auc(s, corte=0.80):
    q = best[best.sinal == s].sort_values("k")
    return int((q.auc.round(2) >= corte).cumprod().sum())
ANT_AUC = {s: lead_auc(s) for s in SINAIS}

fig = go.Figure()
fig.add_hrect(y0=0.8, y1=1.02, fillcolor=CINZA_CLARO, opacity=0.6, line_width=0, layer="below")
fig.add_hline(y=0.5, line=dict(color=CINZA, dash="dot", width=1.5))
fig.add_annotation(x=-6.4, y=0.5, text="0,50 = acaso", showarrow=False, xanchor="left", yanchor="bottom", font=dict(size=11, color=CINZA))
fig.add_annotation(x=-6.4, y=1.0, text="faixa AUC ≥ 0,80", showarrow=False, xanchor="left", yanchor="top", font=dict(size=11, color=CINZA))
for s in ORDEM:
    q = best[best.sinal == s].sort_values("k")
    lab = f"{ROT[s]} ({ROTULO_FORMA[MELHOR[s]].lower()})"
    fig.add_trace(go.Scatter(x=-q.k, y=q.auc, mode="lines+markers", name=lab + (f" · ≥ 0,80 até −{ANT_AUC[s]}" if ANT_AUC[s] else " · < 0,80"),
                             line=dict(color=COR_SINAL[s], width=2), marker=dict(size=8, line=dict(color="white", width=1.5)),
                             customdata=np.c_[q.k, q.n_canc, q.n - q.n_canc],
                             hovertemplate=f"<b>{lab}</b><br>Dados até o mês −%{{customdata[0]}}<br>AUC %{{y:.2f}}"
                                           "<br>%{customdata[1]} cancelados · %{customdata[2]} ativos com dado<extra></extra>"))
fig.update_xaxes(tickvals=list(range(-6, 0)), ticktext=[f"−{k}" for k in range(6, 0, -1)], range=[-6.5, -0.7],
                 title_text="Último mês de dados usado (meses antes da saída)")
fig.update_yaxes(range=[0.35, 1.02], dtick=0.1, tickformat=".1f", title_text="AUC Cancelou × Ativo")
fig.update_layout(height=560, legend=dict(orientation="v", x=1.02, xanchor="left", y=1, yanchor="top"), margin=dict(r=330),
                  title="Quanto tempo de ação cada sinal dá: uso e NPS ainda separam 3 meses antes; os demais só nos 1–2 últimos meses"
                        + sub("Cada ponto usa só dados até o mês indicado (nível/variação da janela de 3 meses que termina ali) · sem vazamento"))
fig.show()

## 3. Limiar e alarme falso

In [5]:
# limiares redondos; vários foram escolhidos olhando as distribuições (ver ressalvas)
LIMIARES = {
    "uso":         Limiar("uso", MELHOR["uso"], -10, "queda ≥ 10 p.p. vs média anterior"),
    "tempo":       Limiar("tempo", MELHOR["tempo"], 28, "≥ 28 h (média 3 meses)"),
    "sla":         Limiar("sla", MELHOR["sla"], 60, "≤ 60% (média 3 meses)"),
    "criticos":    Limiar("criticos", MELHOR["criticos"], 2, "≥ 2/mês (≥ 6 em 3 meses)"),
    "reclamacoes": Limiar("reclamacoes", MELHOR["reclamacoes"], 1, "≥ 1/mês (≥ 3 em 3 meses)"),
    "atraso":      Limiar("atraso", MELHOR["atraso"], 5, "≥ 5 dias (média 3 meses)"),
    "nps":         Limiar("nps", MELHOR["nps"], 6, "última nota ≤ 6 (Detrator)"),
}
assert MELHOR["uso"] == "variacao" and all(MELHOR[s] == "nivel" for s in SINAIS if s != "uso")
H = historico(d, LIMIARES, KS, valores=V)

linhas = []
for s in SINAIS:
    r = desempenho(H, H[s] & H.tem_dados)
    ant = antecedencia_alerta(H, H[s] & H.tem_dados)
    linhas.append(dict(sinal=s, **r, ant_mediana=ant[ST == "Cancelou"][lambda x: x > 0].median()))
TAB = pd.DataFrame(linhas).set_index("sinal").loc[ORDEM]

fig = go.Figure()
ylab = [f"<b>{ROT[s]}</b><br><span style='font-size:11px;color:{CINZA}'>{LIMIARES[s].texto()}</span>" for s in ORDEM]
for s, y in zip(ORDEM, ylab):
    xs = [100 * TAB.loc[s, f"canc_k{k}"] / N_CANC for k in (3, 2, 1)]
    fig.add_trace(go.Scatter(x=xs, y=[y] * 3, mode="lines", line=dict(color=rgba(CORES["Cancelou"], 0.35), width=3),
                             hoverinfo="skip", showlegend=False))
    xa = [100 * TAB.loc[s, "ativos_hoje"] / N_ATIV, 100 * TAB.loc[s, "ativos_algum_mes"] / N_ATIV]
    fig.add_trace(go.Scatter(x=xa, y=[y] * 2, mode="lines", line=dict(color=rgba(CORES["Ativo"], 0.35), width=3),
                             hoverinfo="skip", showlegend=False))
for k, a in [(3, 0.3), (2, 0.6), (1, 1.0)]:
    n = TAB[f"canc_k{k}"]
    fig.add_trace(go.Scatter(x=100 * n / N_CANC, y=ylab, mode="markers", name=f"Cancelados detectados em −{k}",
                             marker=dict(size=13, color=rgba(CORES["Cancelou"], a), line=dict(color=CORES["Cancelou"], width=1.5)),
                             customdata=np.c_[n, [ROT[s] for s in ORDEM]],
                             hovertemplate=f"<b>%{{customdata[1]}}</b><br>Detecta %{{customdata[0]}} de 22 cancelados (%{{x:.0f}}%) no mês −{k}<extra></extra>"))
for col, nome, simb in [("ativos_hoje", "Ativos com alarme hoje (jun/2026)", "square"),
                        ("ativos_algum_mes", "Ativos com alarme em algum mês (últimos 12)", "square-open")]:
    n = TAB[col]
    fig.add_trace(go.Scatter(x=100 * n / N_ATIV, y=ylab, mode="markers", name=nome,
                             marker=dict(size=12, symbol=simb, color=CORES["Ativo"], line=dict(color=CORES["Ativo"], width=2)),
                             customdata=np.c_[n, [ROT[s] for s in ORDEM]],
                             hovertemplate=f"<b>%{{customdata[1]}}</b><br>{nome}: %{{customdata[0]}} de 58 (%{{x:.0f}}%)<extra></extra>"))
fig.update_xaxes(range=[-2, 102], ticksuffix="%", title_text="% do grupo (cancelados: 22 · ativos: 58)")
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=620, margin=dict(l=250, t=170), legend=dict(y=1.04),
                  title="Cada sinal sozinho: quantos cancelados pega e quantos ativos alarma à toa"
                        + sub("Vermelho: % dos 22 cancelados com o sinal disparado em −3, −2, −1 · azul: % dos 58 ativos com o sinal disparado"))
fig.show()

## 4. Piora passageira × afastamento real

In [6]:
MENSAL = {"uso": ("uso_plataforma_pct", 70, -1, "Uso mensal ≤ 70%"), "sla": ("pct_sla_cumprido", 60, -1, "SLA mensal ≤ 60%")}
IDX_C, IDX_A = ST[ST == "Cancelou"].index, ST[ST == "Ativo"].index

def resumo_ep(ep):
    ea, ec = ep[ep.status == "Ativo"], ep[ep.status == "Cancelou"]
    final = ec[ec.em_curso].set_index("cliente_id").duracao.reindex(IDX_C, fill_value=0)
    passado = ea[~ea.em_curso].groupby("cliente_id").duracao.max().reindex(IDX_A, fill_value=0)
    hoje = ea[ea.em_curso].set_index("cliente_id").duracao.reindex(IDX_A, fill_value=0)
    r = {"Ativos que já passaram por isso": ea.cliente_id.nunique(),
         "… e se recuperaram (sequência encerrada)": ea[~ea.em_curso].cliente_id.nunique(),
         "Ativos nessa situação hoje": int((hoje > 0).sum()), "… há ≥ 2 meses seguidos": int((hoje >= 2).sum()),
         "Cancelados nessa situação em −1": int((final > 0).sum()), "… há ≥ 2 meses seguidos ": int((final >= 2).sum()),
         "Duração mediana — cancelados (sequência até −1)": final[final > 0].median(),
         "Duração mediana — ativos (quedas já encerradas)": ea[~ea.em_curso].duracao.median()}
    return r, final, passado, hoje

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.05, subplot_titles=[v[3] for v in MENSAL.values()])
resumo, med = {}, {}
for c, (s, (col, lim, dr, nome)) in enumerate(MENSAL.items(), start=1):
    resumo[nome], final, passado, hoje = resumo_ep(episodios(d.atendimento, col, lim, dr))
    med[s] = resumo[nome]["Duração mediana — cancelados (sequência até −1)"]
    N = np.arange(1, 8)
    for serie, nm, cor, dash, tot in [(final, "Cancelou: sequência que termina em −1", CORES["Cancelou"], "solid", N_CANC),
                                      (passado, "Ativo: maior queda passada, já recuperada", CORES["Ativo"], "solid", N_ATIV),
                                      (hoje, "Ativo: sequência em curso hoje", CORES["Ativo"], "dot", N_ATIV)]:
        y = [100 * (serie >= n).sum() / tot for n in N]
        fig.add_trace(go.Scatter(x=N, y=y, mode="lines+markers", name=nm, legendgroup=nm, showlegend=c == 1,
                                 line=dict(color=cor, width=2, dash=dash), marker=dict(size=8, line=dict(color="white", width=1.5)),
                                 customdata=[(serie >= n).sum() for n in N],
                                 hovertemplate=f"<b>{nm}</b><br>≥ %{{x}} meses seguidos: %{{customdata}} de {tot} (%{{y:.0f}}%)<extra></extra>"),
                      row=1, col=c)
fig.update_xaxes(title_text="Meses seguidos abaixo do limiar (pelo menos)", dtick=1)
fig.update_yaxes(title_text="% do grupo", ticksuffix="%", range=[0, 102], row=1, col=1)
fig.update_layout(height=470, legend=dict(y=1.1),
                  title="Piora passageira × afastamento: nos ativos a queda dura 1 mês e volta; "
                        f"nos cancelados dura {br(med['uso'], 0)} meses (uso) e {br(med['sla'], 1)} (SLA) até a saída"
                        + sub("Valores mensais brutos · ativos: todo o histórico (jan/2025–jun/2026) · cancelados: sequência que chega ao último mês · durações = medianas"))
fig.show()
COND1 = Regra(LIMIARES, "uso", 2, 1).condicao(H) & H.tem_dados
for nome, disp_ in [("Sinal de uso (queda ≥ 10 p.p., 3 meses)", H["uso"]), ("Condição da regra (uso ou ≥ 2 outros)", COND1)]:
    resumo[nome] = resumo_ep(episodios_sinal(H[H.k <= 12], disp_[H.k <= 12]))[0]
display(pd.DataFrame(resumo).rename_axis(columns="Últimos 12 meses p/ sinais de 3 meses · 18 meses p/ valores mensais"))

Últimos 12 meses p/ sinais de 3 meses · 18 meses p/ valores mensais,Uso mensal ≤ 70%,SLA mensal ≤ 60%,"Sinal de uso (queda ≥ 10 p.p., 3 meses)",Condição da regra (uso ou ≥ 2 outros)
Ativos que já passaram por isso,44.0,49.0,23.0,42.0
… e se recuperaram (sequência encerrada),42.0,48.0,22.0,41.0
Ativos nessa situação hoje,6.0,11.0,1.0,9.0
… há ≥ 2 meses seguidos,2.0,4.0,0.0,5.0
Cancelados nessa situação em −1,21.0,12.0,17.0,22.0
… há ≥ 2 meses seguidos,17.0,10.0,13.0,22.0
Duração mediana — cancelados (sequência até −1),3.0,2.5,3.0,4.0
Duração mediana — ativos (quedas já encerradas),1.0,1.0,2.0,2.0


In [7]:
PERS = {p: Regra(LIMIARES, "uso", 2, p) for p in (1, 2, 3)}
res = pd.DataFrame({p: desempenho(H, r.alerta(H)) for p, r in PERS.items()}).T
itens = [("canc_k1", "Cancelados detectados<br>em −1", N_CANC, CORES["Cancelou"]), ("canc_k2", "… em −2", N_CANC, CORES["Cancelou"]),
         ("canc_k3", "… em −3", N_CANC, CORES["Cancelou"]), ("ativos_hoje", "Ativos em alarme<br>hoje", N_ATIV, CORES["Ativo"]),
         ("ativos_algum_mes", "Ativos com alarme em<br>algum mês (12 meses)", N_ATIV, CORES["Ativo"])]
fig = go.Figure()
for p, a in zip(PERS, (0.35, 0.65, 1.0)):
    fig.add_trace(go.Bar(x=[i[1] for i in itens], y=[100 * res.loc[p, i[0]] / i[2] for i in itens],
                         name=f"{p} {'mês' if p == 1 else 'meses seguidos'}", marker=dict(color=[rgba(i[3], a) for i in itens],
                         line=dict(color=[i[3] for i in itens], width=1.5)),
                         text=[f"{int(res.loc[p, i[0]])}" for i in itens], textposition="outside", cliponaxis=False,
                         customdata=[[int(res.loc[p, i[0]]), i[2]] for i in itens],
                         hovertemplate=f"<b>Condição por {p} mês(es) seguido(s)</b><br>%{{x}}: %{{customdata[0]}} de %{{customdata[1]}} (%{{y:.0f}}%)<extra></extra>"))
fig.update_yaxes(range=[0, 110], ticksuffix="%", title_text="% do grupo")
fig.update_layout(barmode="group", bargap=0.25, bargroupgap=0.08, height=450,
                  legend=dict(title_text="Exigir a condição por:"),
                  title="Exigir 2 meses seguidos corta alarmes nos ativos e mantém os 22 cancelados detectados em −1"
                        + sub(f"Condição: {PERS[1].descricao()} · rótulo = nº de clientes · meses de alarme nos ativos (12m): "
                              + " → ".join(f"{br(res.loc[p, 'meses_alarme_ativos_pct'], 0)}%" for p in PERS)))
fig.show()

## 5. Combinação de sinais

In [8]:
fig = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.04,
                    subplot_titles=[f"Dados até o mês −{k}" for k in (1, 2, 3)])
for c, k in enumerate((1, 2, 3), start=1):
    h = H[H.k == k]
    for st in ORDEM_STATUS:
        cnt = h[h.status == st].n_sinais.value_counts().reindex(range(8), fill_value=0)
        tot = cnt.sum()
        fig.add_trace(go.Bar(x=cnt.index, y=100 * cnt / tot, name=st, legendgroup=st, showlegend=c == 1, marker_color=CORES[st],
                             marker_line=dict(color="white", width=1), customdata=cnt,
                             hovertemplate=f"<b>{st}</b> · até −{k}<br>%{{x}} sinais: %{{customdata}} clientes (%{{y:.0f}}%)<extra></extra>"),
                      row=1, col=c)
    c3 = h[h.status == "Cancelou"].n_sinais.ge(3).sum(); a3 = h[h.status == "Ativo"].n_sinais.ge(3).sum()
    fig.layout.annotations[c - 1].text += f"<br><span style='font-size:11px;color:{CINZA}'>≥ 3 sinais: {c3}/22 cancelados · {a3}/58 ativos</span>"
fig.update_xaxes(dtick=1, title_text="Nº de sinais disparados")
fig.update_yaxes(ticksuffix="%", title_text="% do grupo", row=1, col=1)
fig.update_layout(barmode="group", height=450, margin=dict(t=150),
                  title="Quantos sinais disparam ao mesmo tempo: cancelados acumulam vários, a maioria dos ativos fica em 0–1"
                        + sub("7 sinais com os limiares da seção 3"))
fig.show()

In [9]:
val = pd.DataFrame({s: SINAIS[s].direcao * V[1][(s, MELHOR[s])] for s in ORDEM})   # orientado: maior = mais risco
corr = val.corr("spearman")
mask = np.tril(np.ones(corr.shape, dtype=bool), k=-1)
cc = corr.where(mask).iloc[1:, :-1]
fig = go.Figure(go.Heatmap(z=cc.values, x=[ROT[s] for s in cc.columns], y=[ROT[s] for s in cc.index], zmin=-1, zmax=1, xgap=2, ygap=2,
                           colorscale=[[0, "#1c5cab"], [0.5, "#efeeea"], [1, "#b42f2f"]],
                           text=cc.map(lambda v: "" if pd.isna(v) else br(v)).values, texttemplate="%{text}",
                           colorbar=dict(title="ρ", thickness=14),
                           hovertemplate="%{y} × %{x}<br>ρ = %{z:.2f}<extra></extra>"))
fig.update_yaxes(autorange="reversed", showgrid=False)
fig.update_xaxes(tickangle=-30, showgrid=False)
fig.update_layout(height=560, width=860,
                  title="Redundância entre os sinais (80 clientes, dados até −1)"
                        + sub("Spearman dos valores na melhor forma, orientados para risco · alto = contam a mesma história"))
fig.show()

In [10]:
v1 = V[1]
dp1 = disparos(v1, LIMIARES)[ORDEM]
canc = dp1[ST == "Cancelou"].copy()
perdidos = canc.index[~canc["uso"]]
outros = canc.drop(columns="uso").sum(axis=1)
canc = canc.assign(_p=~canc["uso"], _n=canc.sum(axis=1)).sort_values(["_p", "_n"], ascending=[True, False]).drop(columns=["_p", "_n"])
valtxt = pd.DataFrame({s: v1.loc[canc.index, (s, MELHOR[s])].map(lambda x: "" if pd.isna(x) else br(x, 1)) for s in ORDEM})
ylab = [f"<b>{c}</b> ← uso não pega" if c in perdidos else c for c in canc.index]
fig = go.Figure(go.Heatmap(z=canc.astype(int).values, x=[ROT[s] for s in ORDEM], y=ylab, zmin=0, zmax=1, xgap=2, ygap=2, showscale=False,
                           colorscale=[[0, "#f3f2ee"], [1, CORES["Cancelou"]]], text=valtxt.values, texttemplate="%{text}",
                           customdata=np.dstack([valtxt.values, np.tile([LIMIARES[s].texto() for s in ORDEM], (len(canc), 1))]),
                           hovertemplate="<b>%{y}</b> · %{x}<br>Valor: %{customdata[0]}<br>Limiar: %{customdata[1]}<extra></extra>"))
fig.update_yaxes(autorange="reversed", showgrid=False, tickfont_size=11)
fig.update_xaxes(side="top", showgrid=False)
comb = canc.loc[perdidos].drop(columns="uso")
freq = comb.sum().sort_values(ascending=False)
fig.update_layout(height=780, margin=dict(l=170, t=190),
                  title=f"Os {len(perdidos)} cancelados que a queda de uso não pega: todos têm ≥ {int(outros[perdidos].min())} outros sinais em −1"
                        + sub("Vermelho = sinal disparado no mês −1 · número = valor na melhor forma (uso em p.p. de variação) · "
                              "mais frequentes entre eles: " + ", ".join(f"{ROT[s]} {int(n)}/{len(perdidos)}" for s, n in freq.head(4).items())))
fig.show()

In [11]:
cands = [Regra(LIMIARES, None, 2, 1), Regra(LIMIARES, None, 3, 1), Regra(LIMIARES, None, 2, 2),
         Regra(LIMIARES, "uso", 2, 1), Regra(LIMIARES, "uso", 3, 1), Regra(LIMIARES, "uso", 2, 2), Regra(LIMIARES, "uso", 3, 2)]
REGRA = Regra(LIMIARES, "uso", 2, 2)
rows = []
for r in cands:
    x = desempenho(H, r.alerta(H))
    rows.append([("▶ " if r == REGRA else "") + r.descricao(), *(f"{x[f'canc_k{k}']}/22" for k in (1, 2, 3)),
                 br(x["antecedencia_mediana"], 1), f"{x['ativos_hoje']}/58", f"{x['ativos_algum_mes']}/58", f"{br(x['meses_alarme_ativos_pct'], 1)}%"])
cab = ["Regra", "Cancelados<br>em −1", "em −2", "em −3", "Antecedência<br>mediana (meses)", "Ativos em<br>alarme hoje",
       "Ativos c/ alarme<br>em algum mês (12m)", "Meses em alarme<br>nos ativos"]
cores = [["#fbeaea" if r[0].startswith("▶") else "white" for r in rows]] * len(cab)
fig = go.Figure(go.Table(columnwidth=[3.6] + [1] * 7,
                         header=dict(values=[f"<b>{c}</b>" for c in cab], fill_color="#f3f2ee", align="left", font_size=12),
                         cells=dict(values=list(map(list, zip(*rows))), fill_color=cores, align="left", height=28, font_size=12)))
fig.update_layout(height=420, margin=dict(l=10, r=10, t=90, b=10),
                  title="Combinações de sinais: detecção nos 22 cancelados × alarmes nos 58 ativos (▶ = regra proposta)")
fig.show()

## 6. Valor em jogo

In [12]:
H["alerta"] = REGRA.alerta(H).values
H["condicao"] = REGRA.condicao(H) & H.tem_dados
receita_canc = cli.loc[ST == "Cancelou", "valor_mensal"]
fig = go.Figure()
lin = []
for k in (3, 2, 1):
    a = H[(H.k == k) & (H.status == "Cancelou")].set_index("cliente_id").alerta
    det, nao = receita_canc[a.reindex(receita_canc.index, fill_value=False)], receita_canc[~a.reindex(receita_canc.index, fill_value=False)]
    lin.append((f"Alerta até −{k}", det, nao))
for nome, cor, idx_ in [("Detectados pela regra", CORES["Cancelou"], 1), ("Não detectados", "#d6d4ce", 2)]:
    fig.add_trace(go.Bar(y=[l[0] for l in lin], x=[l[idx_].sum() for l in lin], orientation="h", name=nome, marker_color=cor,
                         marker_line=dict(color="white", width=2),
                         text=[f"{fmt_brl(l[idx_].sum())} · {len(l[idx_])} clientes" if len(l[idx_]) else "" for l in lin],
                         textposition="inside", insidetextanchor="middle",
                         customdata=[", ".join(l[idx_].sort_values(ascending=False).index) or "—" for l in lin],
                         hovertemplate=f"<b>%{{y}}</b> · {nome}<br>R$ %{{x:,.0f}}/mês<br>%{{customdata}}<extra></extra>"))
fig.update_xaxes(tickprefix="R$ ", tickformat=",.0f", title_text="Receita mensal dos 22 cancelados")
fig.update_layout(barmode="stack", height=330, title=f"Receita perdida ({fmt_brl(receita_canc.sum())}/mês) que a regra teria sinalizado a tempo"
                  + sub(REGRA.descricao()))
fig.show()

In [13]:
hoje = H[(H.k == 1) & (H.status == "Ativo")].set_index("cliente_id")
hoje["evid"] = evidencias(V[1], LIMIARES).reindex(hoje.index)
hoje["meses"] = antecedencia_alerta(H, H.condicao).reindex(hoje.index)
cat = np.select([hoje.alerta, hoje.condicao, hoje.n_sinais >= 1], ["Alerta (regra)", "Condição só no último mês", "1 sinal isolado"], "Sem sinal")
hoje["cat"] = cat
CAT = {"Alerta (regra)": CORES["Cancelou"], "Condição só no último mês": AMBAR, "1 sinal isolado": rgba(CORES["Ativo"], 0.9), "Sem sinal": "#c9c7c0"}
rng = np.random.default_rng(7)
fig = go.Figure()
for nome, cor in CAT.items():
    s = hoje[hoje.cat == nome]
    if s.empty:
        continue
    rotulo = nome in ("Alerta (regra)", "Condição só no último mês")
    fig.add_trace(go.Scatter(x=s.n_sinais + rng.uniform(-0.18, 0.18, len(s)), y=s.valor_mensal, mode="markers+text" if rotulo else "markers",
                             name=f"{nome} ({len(s)})", text=s.index if rotulo else None, textposition="top center", textfont_size=11,
                             marker=dict(color=cor, size=np.clip(8 + 4 * s.n_sinais, 8, 30), line=dict(color="white", width=1.5), opacity=0.9),
                             customdata=np.c_[s.index, s.n_sinais, s.evid.map(lambda e: "<br>".join(e) if e else "—"),
                                              [cli.loc[c, "segmento"] + " · " + str(cli.loc[c, "plano"]) for c in s.index]],
                             hovertemplate="<b>%{customdata[0]}</b> · %{customdata[3]}<br>R$ %{y:,.0f}/mês<br>%{customdata[1]} sinais:<br>"
                                           "%{customdata[2]}<extra></extra>"))
em = hoje[hoje.alerta]
fig.update_xaxes(title_text="Nº de sinais disparados hoje (jun/2026)", dtick=1, range=[-0.5, 7.5])
fig.update_yaxes(title_text="Valor mensal do contrato (R$)", tickprefix="R$ ", tickformat=",.0f", type="log")
fig.update_layout(height=620, legend=dict(y=1.04),
                  title=f"Ativos hoje: {len(em)} em alerta somam {fmt_brl(em.valor_mensal.sum())}/mês "
                        f"({br(100 * em.valor_mensal.sum() / hoje.valor_mensal.sum(), 1)}% da receita ativa)"
                        + sub(f"Regra: {REGRA.descricao()} · tamanho da bolha = nº de sinais · eixo do valor em escala log"))
fig.show()

fila = (hoje[hoje.alerta | hoje.condicao]
        .assign(Situação=lambda x: x.cat, Segmento=lambda x: cli.loc[x.index, "segmento"], Plano=lambda x: cli.loc[x.index, "plano"].astype(str),
                **{"Valor mensal": lambda x: x.valor_mensal.map(fmt_brl), "Nº de sinais": lambda x: x.n_sinais,
                   "Meses seguidos com a condição": lambda x: x.meses.astype(int), "Sinais disparados (valor em −1)": lambda x: x.evid.map("; ".join)})
        .sort_values(["alerta", "valor_mensal"], ascending=False)
        [["Situação", "Segmento", "Plano", "Valor mensal", "Nº de sinais", "Meses seguidos com a condição", "Sinais disparados (valor em −1)"]]
        .rename_axis("Cliente"))
display(fila.style.set_properties(**{"text-align": "left"}).set_properties(subset=["Sinais disparados (valor em −1)"], **{"white-space": "normal", "max-width": "520px"}))

,Situação,Segmento,Plano,Valor mensal,Nº de sinais,Meses seguidos com a condição,Sinais disparados (valor em −1)
Cliente,,,,,,,
C080,Alerta (regra),Varejo,Enterprise,R$ 20.924,3,2,"Uso da plataforma: −11,7 p.p. vs média anterior; Atraso de pagamento: 9,3 dias; Nota NPS: 3,0"
C067,Alerta (regra),Logística,Avançado,R$ 8.672,5,7,"SLA cumprido: 46,3%; Chamados críticos: 2,0 /mês; Reclamações formais: 1,0 /mês; Atraso de pagamento: 10,0 dias; Nota NPS: 2,0"
C064,Alerta (regra),Logística,Essencial,R$ 4.226,2,2,"SLA cumprido: 53,3%; Nota NPS: 6,0"
C016,Alerta (regra),Logística,Essencial,R$ 2.940,2,10,"Atraso de pagamento: 10,0 dias; Nota NPS: 2,0"
C060,Alerta (regra),Serviços,Essencial,R$ 2.677,4,3,"Tempo médio de resolução: 31,3 h; SLA cumprido: 49,0%; Reclamações formais: 1,3 /mês; Atraso de pagamento: 6,7 dias"
C011,Condição só no último mês,Varejo,Enterprise,R$ 30.742,2,1,"Chamados críticos: 2,0 /mês; Atraso de pagamento: 5,0 dias"
C078,Condição só no último mês,Educação,Essencial,R$ 3.023,2,1,"Atraso de pagamento: 7,7 dias; Nota NPS: 5,0"
C042,Condição só no último mês,Saúde,Essencial,R$ 2.908,2,1,"Reclamações formais: 1,0 /mês; Nota NPS: 6,0"
C019,Condição só no último mês,Saúde,Essencial,R$ 2.897,2,1,"SLA cumprido: 44,4%; Reclamações formais: 1,0 /mês"


## 7. Tabela final dos sinais

In [14]:
def linha(nome, forma, limiar, auc1, auc3, ant_auc, r, ant):
    return {"Sinal": nome, "Melhor forma": forma, "Limiar sugerido": limiar, "AUC (−1)": auc1, "AUC (−3)": auc3,
            "AUC ≥ 0,80 até": ant_auc, "Antecedência típica (meses)": ant,
            "Cancelados detectados −1": pct(r["canc_k1"], N_CANC), "−2": pct(r["canc_k2"], N_CANC), "−3": pct(r["canc_k3"], N_CANC),
            "Alarme falso hoje": pct(r["ativos_hoje"], N_ATIV), "Ativos c/ alarme em algum mês (12m)": pct(r["ativos_algum_mes"], N_ATIV)}
bq = best.set_index(["sinal", "k"]).auc
FINAL = [linha(ROT[s], ROTULO_FORMA[MELHOR[s]], LIMIARES[s].texto(), br(bq[(s, 1)]), br(bq[(s, 3)]),
               f"−{ANT_AUC[s]}" if ANT_AUC[s] else "—", TAB.loc[s].to_dict(), br(TAB.loc[s, "ant_mediana"], 1)) for s in ORDEM]
rr = desempenho(H, H.alerta)
FINAL.append(linha("Regra combinada", "—", REGRA.descricao(), "—", "—", "—", rr, br(rr["antecedencia_mediana"], 1)))
final = pd.DataFrame(FINAL).set_index("Sinal")
display(final.style.set_properties(**{"text-align": "left"})
             .set_properties(subset=pd.IndexSlice[["Regra combinada"], :], **{"font-weight": "bold", "background-color": "#fbeaea"})
             .set_caption("Sinais relevantes: forma, limiar, antecedência e trade-off detecção × alarme falso "
                          "(antecedência típica = mediana de meses seguidos com o sinal disparado até a saída, entre os detectados)"))

,Melhor forma,Limiar sugerido,AUC (−1),AUC (−3),"AUC ≥ 0,80 até",Antecedência típica (meses),Cancelados detectados −1,−2,−3,Alarme falso hoje,Ativos c/ alarme em algum mês (12m)
Sinal,,,,,,,,,,,
Uso da plataforma,Variação,queda ≥ 10 p.p. vs média anterior,"0,97","0,89",−3,"3,0",77% (17/22),64% (14/22),45% (10/22),2% (1/58),40% (23/58)
Nota NPS,Nível,última nota ≤ 6 (Detrator),"0,92","0,83",−3,"4,5",91% (20/22),86% (19/22),68% (15/22),22% (13/58),69% (40/58)
Tempo médio de resolução,Nível,≥ 28 h (média 3 meses),"0,94","0,71",−2,"2,0",77% (17/22),45% (10/22),32% (7/22),3% (2/58),40% (23/58)
SLA cumprido,Nível,≤ 60% (média 3 meses),"0,87","0,69",−2,"2,0",68% (15/22),50% (11/22),32% (7/22),10% (6/58),45% (26/58)
Chamados críticos,Nível,≥ 2/mês (≥ 6 em 3 meses),"0,84","0,77",−2,"2,0",50% (11/22),32% (7/22),27% (6/22),5% (3/58),24% (14/58)
Reclamações formais,Nível,≥ 1/mês (≥ 3 em 3 meses),"0,84","0,70",−2,"2,0",64% (14/22),50% (11/22),36% (8/22),16% (9/58),50% (29/58)
Atraso de pagamento,Nível,≥ 5 dias (média 3 meses),"0,80","0,67",−1,"2,0",59% (13/22),55% (12/22),32% (7/22),12% (7/58),74% (43/58)
Regra combinada,—,"Uso da plataforma (queda ≥ 10 p.p. vs média anterior) ou ≥ 2 outros sinais, por 2 meses seguidos",—,—,—,"3,0",100% (22/22),77% (17/22),64% (14/22),9% (5/58),62% (36/58)


## 8. Sensibilidade aos limiares

In [15]:
VIZ = {"uso": (-8, -12), "tempo": (26, 30), "sla": (55, 65), "criticos": (1.5, 2.5), "reclamacoes": (0.5, 1.5), "atraso": (4, 6), "nps": (5, 7)}
rows = [["Limiares da seção 3", "—", *(f"{rr[f'canc_k{k}']}/22" for k in (1, 2, 3)), f"{rr['ativos_hoje']}/58", f"{rr['ativos_algum_mes']}/58"]]
for s in ORDEM:
    for x in VIZ[s]:
        lim = dict(LIMIARES) | {s: Limiar(s, MELHOR[s], x)}
        hh = historico(d, lim, KS, valores=V)
        r = desempenho(hh, Regra(lim, REGRA.principal, REGRA.min_outros, REGRA.persistencia).alerta(hh))
        rows.append([ROT[s], lim[s].texto(), *(f"{r[f'canc_k{k}']}/22" for k in (1, 2, 3)), f"{r['ativos_hoje']}/58", f"{r['ativos_algum_mes']}/58"])
cab = ["Sinal alterado", "Novo limiar", "Cancelados em −1", "em −2", "em −3", "Ativos em alarme hoje", "Ativos c/ alarme (12m)"]
fig = go.Figure(go.Table(columnwidth=[2, 2, 1, 1, 1, 1.2, 1.2],
                         header=dict(values=[f"<b>{c}</b>" for c in cab], fill_color="#f3f2ee", align="left", font_size=12),
                         cells=dict(values=list(map(list, zip(*rows))), align="left", height=26, font_size=12,
                                    fill_color=[["#fbeaea"] + ["white"] * (len(rows) - 1)] * len(cab))))
fig.update_layout(height=560, margin=dict(l=10, r=10, t=90, b=10),
                  title="Regra proposta com um limiar de cada vez deslocado para o valor vizinho" + sub(REGRA.descricao()))
fig.show()